# 02 · Money Must Be Safe

**Module 08 · Persistent State & Databases**

A refund moves money. Two things can go wrong, and both have cost real companies real money:

1. **The program stops halfway.** The refund is written but the audit line is not (or the other way round).
2. **Two requests arrive at the same moment.** A customer double-clicks, or the agent retries, and the order is refunded **twice**.

A database has a tool for each problem: a **transaction** and a **UNIQUE rule**.

In [1]:
import sqlite3
import tempfile
import threading
import time
from pathlib import Path

import db

scratch = Path(tempfile.mkdtemp())
db.use_database(scratch / "desk.db")        # a throwaway database
db.reset_db()
print("Ready:", db.count_rows("orders"), "orders")

Ready: 4 orders


## 1. A transaction: all or nothing
Issuing a refund is **two writes**: the refund itself, and the audit line that says who did it. They must be saved together. Here is the helper we use for that:

*From `db.py`:*

```python
@contextmanager
def transaction():
    """Open the database and run the block of code as ONE transaction, then close it.

        with db.transaction() as conn:
            conn.execute("INSERT ...")
            conn.execute("INSERT ...")

    If the block finishes, everything is saved (commit). If anything raises an error, NOTHING is saved (rollback)."""
    conn = connect()
    try:
        with conn:      # sqlite3's own "commit if it worked, roll back if it failed"
            yield conn
    finally:
        conn.close()
```

Watch what happens when something fails **between** the two writes:

In [2]:
try:
    with db.transaction() as conn:
        conn.execute("INSERT INTO refunds (order_id, amount, approved_by, created_at) VALUES ('A1001', 79.0, 'agent', ?)", (db.now(),))
        print("Write 1 done: the refund row was inserted (not saved yet)")
        raise RuntimeError("the power goes out here")      # a failure BETWEEN the two writes
        db.log_event(conn, "refund_issued", order_id="A1001")
except RuntimeError as error:
    print("Failure:", error)

print()
print("Refunds in the database:", db.count_rows("refunds"), "  <- the first write was rolled back")
print("Audit lines            :", db.count_rows("audit"))

Write 1 done: the refund row was inserted (not saved yet)
Failure: the power goes out here

Refunds in the database: 0   <- the first write was rolled back
Audit lines            : 0


No half-refund. Without the transaction we would now have **a refund with no audit line**: money moved, and no record of who allowed it.

> **ACID** is the word you will hear for this: **A**tomic (all or nothing), **C**onsistent (the rules always hold), **I**solated (requests do not trip over each other), **D**urable (saved means saved). A transaction gives you all four.

## 2. A race condition, on purpose
Now the second problem. The tempting way to avoid a double refund is **check, then write**:

> *"Is there already a refund for this order? No. OK, insert one."*

It looks right. It is not, because something else can happen **between** the check and the write. We make that gap visible with a short `sleep` and eight requests at the same moment (like a customer who clicked eight times):

In [3]:
naive_db = scratch / "naive.db"
setup = sqlite3.connect(naive_db)
setup.execute("CREATE TABLE refunds (order_id TEXT, amount REAL)")      # note: NO unique rule
setup.commit()
setup.close()


def naive_refund(order_id):
    """Check, then write. Looks fine, is not."""
    conn = sqlite3.connect(naive_db, timeout=10)
    already = conn.execute("SELECT COUNT(*) FROM refunds WHERE order_id = ?", (order_id,)).fetchone()[0]
    time.sleep(0.05)                  # the gap between the check and the write (a slow network, a busy server...)
    if already == 0:
        conn.execute("INSERT INTO refunds VALUES (?, 79.0)", (order_id,))
        conn.commit()
    conn.close()


threads = []
for number in range(8):
    thread = threading.Thread(target=naive_refund, args=("A1001",))
    threads.append(thread)
    thread.start()
for thread in threads:
    thread.join()

check = sqlite3.connect(naive_db)
count = check.execute("SELECT COUNT(*) FROM refunds WHERE order_id = 'A1001'").fetchone()[0]
check.close()
print("Refunds issued for ONE order:", count)
print("The customer was refunded", count, "times: $" + format(count * 79, ".2f"), "instead of $79.00")

Refunds issued for ONE order: 8
The customer was refunded 8 times: $632.00 instead of $79.00


Every request ran its check **before** any of the others had written, so every one of them saw "no refund yet". This bug is invisible when you test with one request at a time, and it only shows up in production.

## 3. The fix: let the database say no
Instead of checking in Python, we tell the database the rule, once: **`order_id` is UNIQUE** in the `refunds` table. Now it does not matter how many requests arrive or in what order. The database refuses the second one, atomically.

Here is the real `refund_order` from `db.py`. Look for `IntegrityError`:

*From `db.py`:*

```python
def refund_order(order_id, approved_by):
    """Issue a refund. Returns (ok, message). The checks and the writes happen in ONE transaction."""
    order_id = order_id.strip().upper()
    try:
        with transaction() as conn:
            row = conn.execute("SELECT * FROM orders WHERE order_id = ?", (order_id,)).fetchone()
            if row is None:
                return False, "No order found with id " + order_id + "."

            allowed, explanation = refund_rules(dict(row))
            if not allowed:
                return False, "Refund refused. " + explanation

            # Two writes. Either both are saved or neither is.
            conn.execute("INSERT INTO refunds (order_id, amount, approved_by, created_at) VALUES (?, ?, ?, ?)",
                         (order_id, row["price"], approved_by, now()))
            log_event(conn, "refund_issued", order_id=order_id, amount=row["price"], approved_by=approved_by)
            return True, "Refund of $" + format(row["price"], ".2f") + " issued for order " + order_id + "."

    except sqlite3.IntegrityError:
        # The UNIQUE rule on refunds.order_id fired: this order already has a refund.
        # The database says no, even if two requests arrived at the same moment.
        return False, "A refund for order " + order_id + " was already issued."


def request_refund(order_id):
    """What the agent's refund tool calls. Small refunds are issued; large ones wait for a person.
    The $100 limit is in this plain function, NOT in a prompt, so the model cannot talk its way around it."""
    order = get_order(order_id)
    if order is None:
        return "No order found with id " + order_id + "."

    allowed, explanation = refund_rules(order)
    if not allowed:
        return "Refund refused. " + explanation

    if order["refund"] != "none":
        return "Nothing to do: the refund for order " + order["order_id"] + " is already " + order["refund"] + "."

    if order["price"] > AUTO_REFUND_LIMIT:
        return hold_for_approval(order["order_id"], order["price"])

    ok, message = refund_order(order["order_id"], "agent (under the automatic limit)")
    return message


def hold_for_approval(order_id, amount):
    """Put a large refund in the waiting line for a person."""
    try:
        with transaction() as conn:
            conn.execute("INSERT INTO approvals (order_id, amount, status, requested_at) VALUES (?, ?, 'pending', ?)",
                         (order_id, amount, now()))
            log_event(conn, "refund_held", order_id=order_id, amount=amount)
            return "The refund of $" + format(amount, ".2f") + " for order " + order_id + " is above the automatic limit of $" \
                + format(AUTO_REFUND_LIMIT, ".0f") + ". It now waits for a person to approve it."
    except sqlite3.IntegrityError:
        return "The refund for order " + order_id + " is already waiting for a person to approve it."


def approve(order_id, approver):
    """A person approves a waiting refund. Returns (ok, message).
    Three things happen in ONE transaction: the approval is closed, the refund is written, the audit line is written."""
    order_id = order_id.strip().upper()
    try:
        with transaction() as conn:
            waiting = conn.execute("SELECT approval_id, amount FROM approvals WHERE order_id = ? AND status = 'pending'",
                                   (order_id,)).fetchone()
            if waiting is None:
                return False, "No refund is waiting for approval on order " + order_id + "."

            conn.execute("UPDATE approvals SET status = 'approved', decided_by = ?, decided_at = ? WHERE approval_id = ?",
                         (approver, now(), waiting["approval_id"]))
            conn.execute("INSERT INTO refunds (order_id, amount, approved_by, created_at) VALUES (?, ?, ?, ?)",
                         (order_id, waiting["amount"], approver, now()))
            log_event(conn, "refund_approved", order_id=order_id, amount=waiting["amount"], approved_by=approver)
            return True, "Refund of $" + format(waiting["amount"], ".2f") + " approved by " + approver + " for order " + order_id + "."
    except sqlite3.IntegrityError:
        return False, "A refund for order " + order_id + " was already issued."


def reject(order_id, approver):
    """A person says no to a waiting refund. Returns (ok, message)."""
    order_id = order_id.strip().upper()
    with transaction() as conn:
        waiting = conn.execute("SELECT approval_id FROM approvals WHERE order_id = ? AND status = 'pending'",
                               (order_id,)).fetchone()
        if waiting is None:
            return False, "No refund is waiting for approval on order " + order_id + "."

        conn.execute("UPDATE approvals SET status = 'rejected', decided_by = ?, decided_at = ? WHERE approval_id = ?",
                     (approver, now(), waiting["approval_id"]))
        log_event(conn, "refund_rejected", order_id=order_id, rejected_by=approver)
        return True, "Refund for order " + order_id + " rejected by " + approver + "."
```

In [4]:
db.reset_db()
results = []


def try_refund():
    ok, message = db.refund_order("A1001", "agent")
    results.append((ok, message))


threads = []
for number in range(8):
    thread = threading.Thread(target=try_refund)
    threads.append(thread)
    thread.start()
for thread in threads:
    thread.join()

winners = 0
for ok, message in results:
    if ok:
        winners = winners + 1
print("Requests that issued a refund:", winners, "of", len(results))
print("Refunds in the database      :", db.count_rows("refunds"))
print("Audit lines                  :", db.count_rows("audit"))
print()
print("One of the answers the other seven got:", results[-1][1])

Requests that issued a refund: 1 of 8
Refunds in the database      : 1
Audit lines                  : 1

One of the answers the other seven got: A refund for order A1001 was already issued.


Exactly one refund, one audit line, and seven polite refusals.

> **Interview sound bite:** *"Do not check-then-write in application code. Put the rule in the database as a constraint, so it holds even when requests race."* The same idea is called **idempotency**: doing the same request twice has the same effect as doing it once.

## 4. The $100 limit, and the approval line
Refunds over $100 wait for a person. That waiting line is also a table, with its own rule: **at most one pending approval per order** (a partial unique index). Watch what happens when the customer asks three times and then a person approves:

In [5]:
db.reset_db()

print("1st ask:", db.request_refund("A1004"))       # a $250 television
print("2nd ask:", db.request_refund("A1004"))
print("Waiting:", db.list_pending())
print()
print("A person approves:", db.approve("A1004", "Priya")[1])
print("Approving again  :", db.approve("A1004", "Priya")[1])
print("Asking again     :", db.request_refund("A1004"))

1st ask: The refund of $250.00 for order A1004 is above the automatic limit of $100. It now waits for a person to approve it.
2nd ask: Nothing to do: the refund for order A1004 is already waiting for a person to approve.
Waiting: [{'order_id': 'A1004', 'amount': 250.0, 'requested_at': '2026-11-22T21:22:49'}]

A person approves: Refund of $250.00 approved by Priya for order A1004.
Approving again  : No refund is waiting for approval on order A1004.
Asking again     : Nothing to do: the refund for order A1004 is already issued: $250.00 on 2026-11-22T21:22:49.


## 5. The audit trail
Every money event was written **in the same transaction** as the money, so the trail can never disagree with the refunds table:

In [6]:
for line in db.audit_trail():
    print(line["at"], line["event"].ljust(16), line["details"])

2026-11-22T21:22:49 refund_held      {"order_id": "A1004", "amount": 250.0}
2026-11-22T21:22:49 refund_approved  {"order_id": "A1004", "amount": 250.0, "approved_by": "Priya"}


## What you learned
- A **transaction** makes several writes succeed or fail together (`with db.transaction()`).
- A **UNIQUE** rule (and `CHECK`, and foreign keys) makes the **database** enforce the business rules, even under a race.
- Never *check, then write* in application code when two requests can overlap.
- Write the **audit line in the same transaction** as the thing it describes.

**Next:** the desk's memory of its customers.